# Notes: Data Science Pipelines and Implementation Using scikit-learn

## Introduction & Brief History

In modern machine learning workflows, raw data rarely enters an estimator directly. Raw inputs require imputation, categorical encoding, feature scaling, and dimensionality reduction before feeding into an algorithm. Historically, practitioners executed these transformations manually as ad-hoc Python scripts or disjointed cells in Jupyter Notebooks.

This disconnected setup led to two severe engineering challenges:

1. **Data Leakage:** Information from the test or validation dataset inadvertently leaked into feature preprocessing steps (such as calculating the global mean or standard deviation on the entire dataset prior to splitting), producing artificially inflated performance metrics that collapsed in production.
2. **Reproducibility Bottlenecks:** Moving code from experimentation to deployment required fragile, error-prone recasting of preprocessing logic into inference routines.

To resolve this, **scikit-learn** formalized the `Pipeline` abstraction (around release 0.9–0.10 in 2011–2012). It chained multiple transformers alongside a final estimator into a unified Python object that behaves syntactically like a single model.

---

## Conceptual Architecture & Mechanics

A pipeline chains together a sequence of **Transformers** followed by a single terminal **Estimator**.

### The Core Scikit-Learn Interfaces

* **Transformer:** Any class implementing `.fit()` and `.transform()` (or `.fit_transform()`). It learns parameters from training features $X$ and returns transformed representations (e.g., `StandardScaler`, `SimpleImputer`, `OneHotEncoder`).
* **Estimator:** Any class implementing `.fit()` and `.predict()` (or `.predict_proba()`, `.score()`). It learns model weights and produces outputs (e.g., `LogisticRegression`, `RandomForestClassifier`).

### How Execution Works

```
Training Flow (pipe.fit(X_train, y_train)):
X_train ---> [ Transformer 1: fit_transform ] ---> [ Transformer 2: fit_transform ] ---> [ Estimator: fit ]

Inference Flow (pipe.predict(X_test)):
X_test  ---> [ Transformer 1: transform ]     ---> [ Transformer 2: transform ]     ---> [ Estimator: predict ]

```

1. **Calling `.fit(X_train, y_train)`:**
* Step 1 calls `fit_transform(X_train)` on Transformer 1. Learned statistics (e.g., column means $\mu_{train}$) are stored as internal attributes.
* Transformed outputs pass to Transformer 2, which runs `fit_transform()`.
* The final output passes into the terminal Estimator, which executes `.fit(X_transformed, y_train)`.


2. **Calling `.predict(X_test)`:**
* Step 1 calls strictly `.transform(X_test)` using the stored $\mu_{train}$. No re-fitting takes place.
* The transformed array passes through subsequent transformers via `.transform()`.
* The final estimator calls `.predict(X_transformed_test)` to return predictions.

---

## Real-World Applications

* **Credit Risk Assessment:** Financial datasets combine continuous scores (credit utilization, debt-to-income) and categorical flags (employment type, geographic region). Pipelines ensure missing historical records are imputed using baseline train cohorts without cross-temporal contamination.
* **Clinical Health Records:** Medical prediction systems process raw lab measurements that require robust quantile transforms alongside one-hot encoded diagnosis codes. Deploying an exported serialized pipeline file (`.joblib`) ensures real-time patient charts receive exact transformations without manual preprocessing scripts.
* **NLP & Text Classification:** Chaining text extraction (`TfidfVectorizer`), feature reduction (`TruncatedSVD`), and a linear classifier (`LinearSVC`) creates a self-contained text classification module that takes raw text strings as inputs directly.

---

## Advantages & Disadvantages

| Dimension | scikit-learn `Pipeline` | Ad-Hoc Scripting / Custom Objects |
| --- | --- | --- |
| **Data Leakage Risk** | **Virtually Eliminated:** Cross-validation folds re-fit transforms independently within each split. | **High:** Global transformations applied before splitting bleed test set distributions into training metrics. |
| **Code Readability & Cleanliness** | **High:** Replaces dozens of disjointed intermediate variables with a single declarative workflow. | **Low:** Cluttered environments with variable tracking like `X_train_scaled_imputed`. |
| **Hyperparameter Optimization** | **Unified:** Can jointly search preprocessing settings (e.g., imputer strategy) and model settings. | **Manual / Fragile:** Requires manual loops across custom functions inside cross-validation splits. |
| **Serialization & Deployment** | **Seamless:** A single `joblib.dump(pipeline, "model.pkl")` saves all parameters and transformers. | **Complex:** Requires exporting mapping tables, scaler weights, and model files separately. |
| **Flexibility on Non-Tabular Data** | **Moderate:** Rigid input shapes (`2D array-like`); less suited for multimodal streaming data or deep learning tensors. | **High:** Custom Python scripts can handle arbitrary inputs, non-standard graphs, and complex branching. |

---

## Recommended References

* **Documentation:** [scikit-learn Pipeline and FeatureUnion Guide](https://scikit-learn.org/stable/modules/compose.html)
* **Textbook:** Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media. (Chapter 2 covers end-to-end transformation pipelines).
* **Academic Paper:** Buitinck, L., Louppe, G., Blondel, M., et al. (2013). *API design for machine learning software: experiences from the scikit-learn project*. arXiv preprint arXiv:1309.0238.

---


# *sklearn* Pipeline Architecture: Classes and Interfaces

Ref: [scikit-learn Pipeline and FeatureUnion Guide](https://scikit-learn.org/stable/modules/compose.html)

In scikit-learn, data pipelines are built on a consistent object-oriented interface. Every step in a pipeline is an object implementing a standardized contract.

| Interface / Class | Module | Primary Methods | Role & Contract |
| --- | --- | --- | --- |
| **`BaseEstimator`** | `sklearn.base` | `get_params()`, `set_params()` | Base class for all objects. Enables hyperparameter inspection and grid/random searches. |
| **`TransformerMixin`** | `sklearn.base` | `fit()`, `transform()`, `fit_transform()` | Mixin for data transformation steps. `fit_transform()` is auto-implemented via `fit().transform()`. |
| **`RegressorMixin`** / **`ClassifierMixin`** | `sklearn.base` | `score()` | Mixin for terminal estimators. Provides default scoring metrics ($R^2$ for regressors, accuracy for classifiers). |
| **`Pipeline`** | `sklearn.pipeline` | `fit()`, `transform()`, `predict()`, `predict_proba()` | Sequentially chains transformers. All intermediate steps must implement `TransformerMixin`; the final step can be an estimator. |
| **`ColumnTransformer`** | `sklearn.compose` | `fit()`, `transform()`, `fit_transform()` | Applies different transformers to subsets of columns (numeric, categorical, text) and horizontally concatenates outputs. |
| **`FeatureUnion`** | `sklearn.pipeline` | `fit()`, `transform()` | Applies multiple transformers in parallel across the *entire* dataset and concatenates resulting feature spaces. |

---

## The Estimator Contract

To integrate custom logic into a scikit-learn pipeline, classes follow strict behavioral rules:

1. **Estimator**: Must implement `fit(X, y=None)`. Learns internal state (e.g., column means, vocabulary, class distributions) and returns `self`. Learned attributes must have a trailing underscore (e.g., `self.mean_`).
2. **Transformer**: Must implement `transform(X)`. Uses learned parameters from `fit` to transform inputs without updating internal state.
3. **Predictor**: Must implement `predict(X)`. Produces predictions using parameters learned during `fit`.

---

## Key Pipeline Construction Rules

* **Data Leakage Prevention**: Never fit individual transformers prior to calling `Pipeline.fit()`. The `Pipeline` ensures all transformations on `X_train` compute parameters (means, standard deviations, quantile boundaries) solely from training folds, and applies `transform()` without recalibration on `X_test` or validation folds.
* **Handling Unknowns**: `OneHotEncoder(handle_unknown="ignore")` is essential for production inference; unseen discrete labels encountered at runtime will map to all-zero vectors instead of raising `ValueError`.
* **Parameter Access**: Hyperparameters within nested pipelines use double-underscore syntax (e.g., `preprocessor__num__imputer__strategy` or `regressor__n_estimators`), enabling end-to-end tuning through `GridSearchCV` or `RandomizedSearchCV`

---

## End-to-End Implementation

The following complete, self-contained script handles:

* Synthetic tabular data with discrete and continuous features containing missing values (`NaN`).
* Custom domain-specific transformation inheriting from `BaseEstimator` and `TransformerMixin`.
* Median imputation and standard scaling for continuous features.
* Most-frequent imputation and one-hot encoding for categorical/discrete features.
* End-to-end integration via `ColumnTransformer` and `Pipeline` terminating in a `RandomForestRegressor`.
* Model evaluation and complete pipeline serialization/deserialization via `joblib`.

---

In [3]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ==========================================
# 1. SYNTHETIC DATA GENERATION
# ==========================================
np.random.seed(42)
n_samples = 1500

data = {
    # Continuous features
    "sqft": np.random.normal(1800, 500, n_samples).clip(500, 5000),
    "lot_size": np.random.exponential(4000, n_samples).clip(1000, 25000),
    "distance_to_center_km": np.random.uniform(1.0, 30.0, n_samples),
    # Discrete features (categorical / counts)
    "bedrooms": np.random.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.1, 0.25, 0.4, 0.2, 0.05]),
    "property_type": np.random.choice(
        ["Detached", "Semi-Detached", "Townhouse", "Condo"],
        size=n_samples,
        p=[0.45, 0.25, 0.20, 0.10],
    ),
    "neighborhood_tier": np.random.choice(["Tier_1", "Tier_2", "Tier_3"], size=n_samples),
}

df = pd.DataFrame(data)

# Inject synthetic target: Price
df["price"] = (
    df["sqft"] * 150
    + df["bedrooms"] * 15000
    - df["distance_to_center_km"] * 3500
    + (df["property_type"] == "Detached") * 45000
    + np.random.normal(0, 15000, n_samples)
)

# Inject random missing values (NaNs) across both continuous and discrete columns
df.loc[df.sample(frac=0.06, random_state=1).index, "sqft"] = np.nan
df.loc[df.sample(frac=0.04, random_state=2).index, "lot_size"] = np.nan
df.loc[df.sample(frac=0.05, random_state=3).index, "property_type"] = np.nan
df.loc[df.sample(frac=0.03, random_state=4).index, "bedrooms"] = np.nan


# ==========================================
# 2. CUSTOM TRANSFORMER DEFINITION (advanced action)
# ==========================================
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Computes custom domain interactions:
    e.g., ratio of square footage to bedrooms.
    """

    def __init__(self, add_sqft_per_bedroom: bool = True):
        self.add_sqft_per_bedroom = add_sqft_per_bedroom

    def fit(self, X, y=None):
        # Stateless transformer; nothing to learn
        return self

    def transform(self, X):
        X_out = X.copy()
        if self.add_sqft_per_bedroom:
            # Avoid divide-by-zero errors
            bedrooms = np.where(X_out["bedrooms"] == 0, 1, X_out["bedrooms"])
            X_out["sqft_per_bedroom"] = X_out["sqft"] / bedrooms
        return X_out


# ==========================================
# 3. SUB-PIPELINES & COLUMN TRANSFORMER
# ==========================================
continuous_features = ["sqft", "lot_size", "distance_to_center_km", "sqft_per_bedroom"]
discrete_features = ["bedrooms", "property_type", "neighborhood_tier"]

# Numerical Sub-Pipeline: Median Imputation -> Standardization
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Categorical/Discrete Sub-Pipeline: Mode Imputation -> One-Hot Encoding
cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,  # Dense array avoids sparse matrix overhead in RF
            ),
        ),
    ]
)

# Column-level aggregation
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, continuous_features),
        ("cat", cat_pipeline, discrete_features),
    ],
    remainder="drop",  # Drops any unlisted columns
    verbose_feature_names_out=False,
)

# ==========================================
# 4. FULL ESTIMATION PIPELINE
# ==========================================
full_pipeline = Pipeline(
    steps=[
        ("feature_engineering", FeatureEngineer(add_sqft_per_bedroom=True)),
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
                        n_estimators=150,
                        max_depth=12,
                        min_samples_leaf=4,
                        random_state=42,
                        n_jobs=-1,
            ),
        ),
    ]
)

# ==========================================
# 5. TRAINING AND EVALUATION
# ==========================================
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit end-to-end pipeline (Feature Engineering -> Preprocessing -> Training)
full_pipeline.fit(X_train, y_train)

# Evaluation
y_pred = full_pipeline.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = full_pipeline.score(X_test, y_test)

print(f"Test RMSE: ${rmse:,.2f}")
print(f"Test MAE:  ${mae:,.2f}")
print(f"Test R^2:  {r2:.4f}")

# Inspect transformed feature names generated dynamically
feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()
print(f"\nEngineered feature count: {len(feature_names)}")
print("Sample encoded features:", feature_names[:6])

# ==========================================
# 6. SERIALIZATION AND INFERENCE
# ==========================================

# Serialize (save) the created ML model into a file for subsequent use / deployment

MODEL_ARTIFACT_PATH = "housing_rf_pipeline.joblib"

# Serialize entire pipeline including learned stats and trained trees
joblib.dump(full_pipeline, MODEL_ARTIFACT_PATH, compress=3)
print(f"\nSerialized pipeline artifact saved to: {MODEL_ARTIFACT_PATH}")

# Reload model from disk
loaded_pipeline = joblib.load(MODEL_ARTIFACT_PATH)

# Simulate unseen production batch with missing inputs
raw_inference_payload = pd.DataFrame(
    [
        {
            "sqft": 2400.0,
            "lot_size": 5200.0,
            "distance_to_center_km": 4.5,
            "bedrooms": 3.0,
            "property_type": "Detached",
            "neighborhood_tier": "Tier_1",
        },
        {
            "sqft": np.nan,  # Relies on learned median imputer
            "lot_size": 3100.0,
            "distance_to_center_km": 14.2,
            "bedrooms": np.nan,  # Relies on learned mode imputer
            "property_type": "Condo",
            "neighborhood_tier": "Tier_3",
        },
    ]
)

predictions = loaded_pipeline.predict(raw_inference_payload)
print("\nInference on raw production payloads:")
for idx, pred in enumerate(predictions):
    print(f" Sample {idx + 1} Predicted Value: ${pred:,.2f}")

Test RMSE: $28,538.44
Test MAE:  $19,569.06
Test R^2:  0.8923

Engineered feature count: 16
Sample encoded features: ['sqft' 'lot_size' 'distance_to_center_km' 'sqft_per_bedroom'
 'bedrooms_1.0' 'bedrooms_2.0']

Serialized pipeline artifact saved to: housing_rf_pipeline.joblib

Inference on raw production payloads:
 Sample 1 Predicted Value: $414,552.18
 Sample 2 Predicted Value: $261,146.22
